# Lab 2 – Streaming LLM data pipeline (TinyStories + Qwen2.5)

Modified version of the course's `Lab2.ipynb`.

| | Original | This version |
|---|---|---|
| Dataset | WikiText-2 (streamed) | **TinyStories train split, streamed** (2.1M stories, 4 parquet files) |
| Tokenizer | GPT-2 | **Qwen2.5** |
| Document boundaries | none | **`<|endoftext|>` appended after every story** |

Streaming means the 2.1M-story train split is never loaded into memory: stories are read, tokenized
and packed into blocks one at a time.

In [1]:
# !pip install -r requirements.txt

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch

In [3]:
# ============================================================
# 1. Load the dataset in STREAMING mode
# ============================================================
# Returns an IterableDataset: nothing is downloaded up front, examples arrive on demand.
stream_dataset = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
print(stream_dataset)
print(f"Underlying files (shards): {stream_dataset.n_shards}")

IterableDataset({
    features: ['text'],
    num_shards: 4
})
Underlying files (shards): 4


In [4]:
# ============================================================
# 2. Initialize the Qwen2.5 tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
EOS_ID = tokenizer.eos_token_id

In [5]:
# ============================================================
# 3. Tokenization step (lazy)
# ============================================================
# No padding/truncation: keep raw token sequences so they can be concatenated later.
# Append <|endoftext|> so story boundaries survive concatenation.
def tokenize_function(examples):
    out = tokenizer(examples["text"], add_special_tokens=False)
    out["input_ids"] = [ids + [EOS_ID] for ids in out["input_ids"]]
    return out

tokenized_stream = stream_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

In [6]:
# ============================================================
# 4. Rolling buffer for grouping into fixed-length blocks
# ============================================================
# A stream can't be looked ahead into, so leftover tokens stay in the buffer
# and are joined with the next story.
block_size = 128

def group_texts_streaming(dataset_iter, block_size):
    buffer = []
    for example in dataset_iter:
        buffer.extend(example["input_ids"])
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {
                "input_ids": chunk,
                "attention_mask": [1] * block_size,
            }

In [7]:
# ============================================================
# 5. Wrap the generator in a PyTorch IterableDataset
# ============================================================
class StreamingLMIterableDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, block_size):
        self.dataset = hf_iterable_dataset
        self.block_size = block_size

    def __iter__(self):
        return group_texts_streaming(self.dataset, self.block_size)

grouped_iterable_dataset = StreamingLMIterableDataset(tokenized_stream, block_size)

In [8]:
# ============================================================
# 6. Collate function for batches
# ============================================================
def collate_fn(batch):
    input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone(),
    }

In [9]:
# ============================================================
# 7. DataLoader for streaming data
# ============================================================
train_loader = DataLoader(grouped_iterable_dataset, batch_size=8, collate_fn=collate_fn)

In [10]:
# ============================================================
# 8. Iterate over a few batches
# ============================================================
print("Sample streaming batches:")
for i, batch in enumerate(train_loader):
    print(f"Batch {i} -> input_ids shape: {batch['input_ids'].shape}")
    if i == 2:
        break

print("\nFirst block of the last batch, decoded:")
print(tokenizer.decode(batch["input_ids"][0]))

Sample streaming batches:


Batch 0 -> input_ids shape: torch.Size([8, 128])
Batch 1 -> input_ids shape: torch.Size([8, 128])
Batch 2 -> input_ids shape: torch.Size([8, 128])

First block of the last batch, decoded:
 pictures every day.<|endoftext|>One day, a girl named Mia went for a walk. She saw a big, scary house. It had a tall door and small windows. Mia was brave, so she went inside the house.

In the house, Mia saw a birdcage. Inside the birdcage, there was a little bird. The bird was sad. It wanted to fly and be free. Mia wanted to help the bird.

Mia opened the birdcage door. The bird flew out and was happy. It was not scary anymore. Mia and the bird were friends. They played and had fun all day.<|endoftext|>Once
